<a href="https://colab.research.google.com/github/juanzegarra/repetitive_sequences/blob/main/transpo_cons_21_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/usr/bin/env python3
import sys
import os
import subprocess
from collections import defaultdict
from Bio import SeqIO
from Bio import AlignIO
from math import log2
from ShannonMSA import shannon_entropy_list_msa, running_mean
import statistics
import argparse
import re

#definicao para biblioteca do argparse
def parse_args():
    parser = argparse.ArgumentParser(
        description="Expand hits around BLAST alignments and compute entropy"
    )

    # Positional (required) arguments
    parser.add_argument("query", help="Query FASTA file")
    parser.add_argument("genome", help="Genome FASTA file")
    parser.add_argument("bp_extend", type=int, help="Base pairs to extend")

    # Optional flags with default values
    parser.add_argument(
        "--outdir",
        default="results",
        help="Output directory (default: results)",
    )
    parser.add_argument(
        "-ec",
        type=float,
        default=1.2,
        help="Entropy cutoff for stopping expansion (default: 1.2)",
    )
    parser.add_argument(
        "--max_cycles",
        type=int,
        default=10,
        help="Maximum number of expansion cycles (default: 10)",
    )
    parser.add_argument(
        "--window",
        type=int,
        default=20,
        help="Window size for running mean entropy (default: 20)",
    )
    parser.add_argument(
        "--cluster_thres",
        type=float,
        default=0.8,
        help="Threshold for clustering (default: 0.8)",
    )
    parser.add_argument(
        "--cluster_kmer",
        type=int,
        default=5,
        help="K-mer size for clustering (default: 5)",
    )

    return parser.parse_args()


class Query:
    def __init__(self, record, db_file, exp, genome_file, out_dir="results"):
        self.record = record.id
        self.db_file = db_file
        self.exp = exp
        self.expand_cycles = 1
        self.query = f"{record.id}.fasta"
        self.results_tsv = os.path.join(out_dir, f"hits_{self.record}.tsv")
        self.results_bed = os.path.join(
            out_dir, f"hits_{self.record}_round{self.expand_cycles}.bed"
        )
        self.genome_file = genome_file
        self.fasta_exp_hits = os.path.join(
            out_dir, f"{self.record}_exp_hits_round{self.expand_cycles}.fa"
        )
        self.round_dir = os.path.join(out_dir, f"results_round{self.expand_cycles}")
        os.makedirs(self.round_dir, exist_ok=True)
        self.alignment = os.path.join(self.round_dir, f"{self.record}_round{self.expand_cycles}.afa")
        self.entropy = None
        self.out_dir = out_dir
        self.clusters = defaultdict(list)


    def run_blast(self):
        print(f"Running BLAST for {self.record}...")
        os.makedirs(self.out_dir, exist_ok=True)

        subprocess.run(
            [
                "blastn",
                "-query",
                self.query,
                "-db",
                self.db_file,
                "-outfmt",
                "6 qseqid sseqid sstart send",
                "-out",
                self.results_tsv,
            ]
        )

    ##function for expanding blast hits
    def expand_results(self):
        print(f"Round: {self.expand_cycles}. Expanding hits...")
        with open(self.results_tsv, "r") as fin, open(
            self.results_bed, "w"
        ) as fout:
            for line in fin:
                qseqid, sseqid, sstart, send = line.strip().split()
                sstart, send = int(sstart), int(send)

                # normalize strand orientation
                start, end = min(sstart, send), max(sstart, send)

                # expand window depending on cycle
                expand_size = self.exp * self.expand_cycles
                start = max(0, start - expand_size)  # avoid negative coords
                end = end + expand_size

                # BED requires 0-based start
                fout.write(f"{sseqid}\t{start - 1}\t{end}\n")

        self.expand_cycles += 1

        subprocess.run(
            [
                "bedtools",
                "getfasta",
                "-fi",
                self.genome_file,
                "-bed",
                self.results_bed,
                "-fo",
                self.fasta_exp_hits,
            ]
        )


    ##using muscle for msa
    def mult_align(self, out_dir="results"):
        print("Aligning copies...")
        input_aln = self.fasta_exp_hits
        if os.path.isfile(input_aln):
            subprocess.run(["muscle", "-align", input_aln, "-output", self.alignment, "--quiet",])
        else:
            print("Error: no input file for MSA")

    ## Shannon entropy call function for msa
    def compute_alignment_score(self, msa_file, fmt="fasta", window=20):
        alignment = AlignIO.read(msa_file, fmt)
        entropies = shannon_entropy_list_msa(alignment)

        if window > 0:
            entropies = running_mean(entropies, window)

        self.entropy = statistics.median(entropies)
        print(
            f"Entropy for alignment round {self.expand_cycles} is {self.entropy}"
        )

        return self.entropy

    def cluster_seqs(self, out_dir=None, cluster_thres=0.8, cluster_kmer=5):

        if out_dir is None:
            out_dir = self.out_dir
        os.makedirs(out_dir, exist_ok=True)

        out_prefix = os.path.join(out_dir, f"clusters_{self.record}")
        clstr_file = out_prefix + ".clstr"
        # run cd-hit-est (nucleotide)
        subprocess.run(
            [
                "cd-hit-est",
                "-i",
                self.fasta_exp_hits,
                "-o",
                out_prefix,
                "-c",
                str(cluster_thres),
                "-n",
                str(cluster_kmer),
                "-d",
                "0",
            ],
            stdout=subprocess.DEVNULL,
            check=True,
        )

        # parse .clstr
        clusters = defaultdict(list)
        current = None
        # regex to extract >seqid... part
        re_seq = re.compile(r">(.+?)\.\.\.")
        with open(clstr_file) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                if line.startswith(">Cluster"):
                    current = line.split()[1]
                    continue
                m = re_seq.search(line)
                if m and current is not None:
                    seqid = m.group(1)
                    clusters[current].append(seqid)
        # store in object and return
        self.clusters = clusters
        return clusters

    def start_clustering(self, cluster_map=None, out_dir=None):

        if out_dir is None:
            out_dir = self.out_dir
        if cluster_map is None:
            cluster_map = self.clusters

        cluster_objects = []
        for cid, seq_ids in cluster_map.items():
            # seq_ids are strings (headers). We will map them to SeqRecord objects:
            seq_records = []
            # build quick dict from fasta_exp_hits
            records = SeqIO.to_dict(SeqIO.parse(self.fasta_exp_hits, "fasta"))
            for sid in seq_ids:
                if sid in records:
                    seq_records.append(records[sid])
                else:
                    # warn if missing
                    print(f"Warning: {sid} not found in {self.fasta_exp_hits}")
            c = Cluster(
                cid,
                seq_records,
                self.genome_file,
                start_cycle=self.expand_cycles,
                out_dir=out_dir,
                parent=self,
            )
            cluster_objects.append(c)
        # store for later
        self.cluster_objects = cluster_objects
        print("starting to cluster...")
        return cluster_objects


class Cluster:
    def __init__(
        self,
        cluster_id,
        seq_records,
        genome_file,
        start_cycle=1,
        out_dir="results",
        parent=None,
    ):
        self.cluster_id = str(cluster_id)
        self.seqs = seq_records  # list of SeqRecord objects
        self.genome_file = genome_file
        self.expand_cycles = int(start_cycle)
        self.out_dir = out_dir
        self.parent = parent  # Query object
        self.bed_file = os.path.join(
            out_dir, f"{self.cluster_id}_round{self.expand_cycles}.bed"
        )
        self.fasta_file = os.path.join(
            out_dir, f"{self.cluster_id}_round{self.expand_cycles}.fa"
        )
        self.fasta_exp_hits = os.path.join(
            out_dir, f"{self.cluster_id}_round{self.expand_cycles}_exp.fa"
        )
        self.round_dir = os.path.join(self.out_dir, f"results_round{self.expand_cycles}_C{self.cluster_id}")
        os.makedirs(self.round_dir, exist_ok=True)
        self.alignment = os.path.join(self.round_dir, f"{self.cluster_id}_round{self.expand_cycles}.afa")
        self.entropy = None
        os.makedirs(out_dir, exist_ok=True)

    #writes fasta file from sequences in each cluster
    def write_fasta(self):
        SeqIO.write(self.seqs, self.fasta_file, "fasta")
        return self.fasta_file

    #creates a bed file based on the header of sequences in fasta file
    def write_cluster_bed_from_headers(self):

        with open(self.fasta_file) as fh, open(self.bed_file, "w") as bedfh:
            for line in fh:
                if not line.startswith(">"):
                    continue
                header = line[1:].strip()
                first = header.split()[0]
                if ":" not in first or "-" not in first:
                    print(
                        f"Warning: header {first} not in contig:start-end format"
                    )
                    continue
                #Initial expand
                chrom, coords = first.split(":", 1)
                start_s, end_s = coords.split("-", 1)
                start1, end1 = int(start_s), int(end_s)
                bp = self.parent.exp if self.parent else 0
                expand_size = bp * self.expand_cycles
                new_start1 = max(1, start1 - expand_size)
                new_end1 = end1 + expand_size
                bed_start = new_start1 - 1
                bedfh.write(f"{chrom}\t{bed_start}\t{new_end1}\n")
        return self.bed_file

    # Uses bedtools get fasta to extract expanded seqs
    def bedtools_getfa(self):
        print("Extracting sequences...")



        subprocess.run(
            [
                "bedtools",
                "getfasta",
                "-fi",
                self.genome_file,
                "-bed",
                self.bed_file,
                "-fo",
                self.fasta_exp_hits
            ],
            check=True,
        )


        return self.fasta_exp_hits
    #extend loop
    def expanding_cluster(self, bp_extend=None):

        if bp_extend is None:
            bp_extend = self.parent.exp if self.parent else 0
        # update expand cycles and filenames
        self.expand_cycles += 1
        self.bed_file = os.path.join(
            self.out_dir, f"{self.cluster_id}_round{self.expand_cycles}.bed"
        )
        self.fasta_exp_hits = os.path.join(
            self.out_dir, f"{self.cluster_id}_round{self.expand_cycles}_exp.fa"
        )
        # write bed from current fasta headers (could also read previous bed and expand)
        self.write_cluster_bed_from_headers()
        self.bedtools_getfa()
    #MUSCLE alignment
    def mult_align(self):
        print(f"Aligning copies from cluster {self.cluster_id}...")
        fasta_in = (
            self.fasta_exp_hits
            if os.path.exists(self.fasta_exp_hits)
            else self.fasta_file
        )
        if not os.path.isfile(fasta_in):
            print(
                f"No fasta for cluster {self.cluster_id}, skipping alignment"
            )
            return None
        subprocess.run(
            ["muscle", "-align", fasta_in, "-output", self.alignment, "--quiet"],
            check=True,
        )
        return self.alignment
    #Uses shannon entropy calculator, extracts mean of whole alignment
    def compute_alignment_score(self, fmt="fasta", window=20):
        print(f"Computing entropy for cluster {self.cluster_id}...")

        if not os.path.exists(self.alignment) or os.path.getsize(self.alignment) == 0:
            print(f"[Warning] Alignment file missing or empty for cluster {self.cluster_id}")
            return None

        alignment = AlignIO.read(self.alignment, fmt)


        if len(alignment) < 2:
            print(f"[Warning] Not enough sequences in cluster {self.cluster_id} to compute entropy")
            self.entropy = 0.0
            return self.entropy

        entropies = shannon_entropy_list_msa(alignment)

        if window > 0 and len(entropies) >= window:
            entropies = running_mean(entropies, window)

        if not entropies:
            self.entropy = 0.0
        else:
            self.entropy = statistics.mean(entropies)

        return self.entropy



def main():
    ##Parsing flags
    args = parse_args()
    query_file = args.query
    genome_file = args.genome
    number_of_bp_to_extend = args.bp_extend
    out_dir = args.outdir
    entropy_cutoff = args.ec
    max_cycles = args.max_cycles
    window = args.window
    cluster_thres = args.cluster_thres
    cluster_kmer = args.cluster_kmer

    work_d = os.getcwd()
    genome_prefix = os.path.splitext(genome_file)[0]

    ##Sanity checking

    if not os.path.isfile(os.path.join(work_d, genome_file)):
        print("Genome file is missing")
    elif not os.path.isfile(os.path.join(work_d, query_file)):
        print("Query file missing")
    else:
        subprocess.run(
            [
                "makeblastdb",
                "-in",
                genome_file,
                "-dbtype",
                "nucl",
                "-out",
                genome_prefix,
            ]
        )

        queries = []

        ##Parsing queries

        for record in SeqIO.parse(query_file, "fasta"):
            query_path = f"{record.id}.fasta"
            SeqIO.write(record, query_path, "fasta")

            q = Query(
                record,
                genome_prefix,
                number_of_bp_to_extend,
                genome_file,
                out_dir,
            )
            # Function calling
            q.run_blast()
            queries.append(q)

            print(f"Results for {record.id} written to {q.results_tsv}")

            q.expand_results()
            q.mult_align()

            if os.path.exists(q.alignment):
                q.compute_alignment_score(q.alignment)

            while q.expand_cycles < max_cycles:
                q.expand_results()
                print(f"Results for {record.id} written to {q.results_bed}")
                q.mult_align()
                print(f"Results for {record.id} written to {q.alignment}")

                if os.path.exists(q.alignment):
                    q.compute_alignment_score(q.alignment)
                    print(f"Entropy for {record.id}: {q.entropy}")
                    if q.entropy < entropy_cutoff:
                        ##Global expansion stops, begin clustering
                        # after q.compute_alignment_score() and q.entropy >= cutoff:
                        cluster_map = q.cluster_seqs(
                            out_dir=q.out_dir,
                            cluster_thres=cluster_thres,
                            cluster_kmer=cluster_kmer,
                        )
                        cluster_objects = q.start_clustering(cluster_map)
                        break

            for c in cluster_objects:
                if len(c.seqs) <= 1:
                    continue

                c.write_fasta()
                c.write_cluster_bed_from_headers()
                c.bedtools_getfa()
                c.mult_align()
                c.compute_alignment_score()

                # expand clusters until entropy falls below cutoff or max cycles reached
                while c.expand_cycles <= max_cycles:
                    if c.entropy is not None and c.entropy <= entropy_cutoff:
                        print(f"Entropy cutoff reached for cluster {c.cluster_id} ({c.entropy}), stopping expansion...")
                        break

                    c.expanding_cluster()
                    c.mult_align()
                    c.compute_alignment_score()
                    print(f"Alignment score for cluster {c.cluster_id} is {c.entropy}")


if __name__ == "__main__":
    main()


In [ ]:
# @title ShannonMSA
# This script will calculate Shannon entropy from a MSA.

# Dependencies:

# Biopython, Matplotlib [optionally], Math

"""
Shannon's entropy equation (latex format):
    H=-\sum_{i=1}^{M} P_i\,log_2\,P_i
    Entropy is a measure of the uncertainty of a probability distribution (p1, ..... , pM)
    https://stepic.org/lesson/Scoring-Motifs-157/step/7?course=Bioinformatics-Algorithms&unit=436
    Where, Pi is the fraction of nuleotide bases of nuleotide base type i,
    and M is the number of nuleotide base types (A, T, G or C)
    H ranges from 0 (only one base/residue in present at that position) to 4.322 (all 20 residues are equally
    represented in that position).
    Typically, positions with H >2.0 are considerered variable, whereas those with H < 2 are consider conserved.
    Highly conserved positions are those with H <1.0 (Litwin and Jores, 1992).
    A minimum number of sequences is however required (~100) for H to describe the diversity of a protein family.
"""
import os
import sys
import warnings
import traceback

__author__ = "Joe R. J. Healey"
__version__ = "1.0.0"
__title__ = "ShannonMSA"
__license__ = "GPLv3"
__author_email__ = "J.R.J.Healey@warwick.ac.uk"


def parseArgs():
    """Parse command line arguments"""

    import argparse

    try:
        parser = argparse.ArgumentParser(
            description='Compute per base/residue Shannon entropy of a Multiple Sequence Alignment.')

        parser.add_argument('-a',
                            '--alignment',
                            action='store',
                            required=True,
                            help='The multiple sequence alignment (MSA) in any of the formats supported by Biopython\'s AlignIO.')
        parser.add_argument('-f',
                            '--alnformat',
                            action='store',
                            default='fasta',
                            help='Specify the format of the input MSA to be passed in to AlignIO.')
        parser.add_argument('-v',
                            '--verbose',
                            action='count',
                            default=0,
                            help='Verbose behaviour, printing parameters of the script.')
        parser.add_argument('-m',
                            '--runningmean',
                            action='store',
                            type=int,
                            default=0,
                            help='Return the running mean (a.k.a moving average) of the MSAs Shannon Entropy. Makes for slightly smoother plots. Providing the number of points to average over switches this on.')
        parser.add_argument('--makeplot',
                            action='store_true',
                            help='Plot the results via Matplotlib.')
    except:
        print "An exception occurred with argument parsing. Check your provided options."
        traceback.print_exc()

    return parser.parse_args()



def parseMSA(msa, alnformat, verbose):
    """Parse in the MSA file using Biopython's AlignIO"""

    from Bio import AlignIO
    alignment = AlignIO.read(msa, alnformat)

    # Do a little sanity checking:
    seq_lengths_list = []
    for record in alignment:
       seq_lengths_list.append(len(record))

    seq_lengths = set(seq_lengths_list)

    if verbose > 0: print("Alignment length is:" + str(list(seq_lengths)))

    if len(seq_lengths) != 1:
        sys.stderr.write("Your alignment lengths aren't equal. Check your alignment file.")
        sys.exit(1)

    index = range(1, list(seq_lengths)[0]+1)

    return alignment, list(seq_lengths), index

##################################################################
# Function to calcuate the Shannon's entropy per alignment column
# H=-\sum_{i=1}^{M} P_i\,log_2\,P_i (http://imed.med.ucm.es/Tools/svs_help.html)
# Gaps and N's are included in the calculation
##################################################################

def shannon_entropy(list_input):
    """Calculate Shannon's Entropy per column of the alignment (H=-\sum_{i=1}^{M} P_i\,log_2\,P_i)"""

    import math
    unique_base = set(list_input)
    M   =  len(list_input)
    entropy_list = []
    # Number of residues in column
    for base in unique_base:
        n_i = list_input.count(base) # Number of residues of type i
        P_i = n_i/float(M) # n_i(Number of residues of type i) / M(Number of residues in column)
        entropy_i = P_i*(math.log(P_i,2))
        entropy_list.append(entropy_i)

    sh_entropy = -(sum(entropy_list))

    return sh_entropy


def shannon_entropy_list_msa(alignment):
    """Calculate Shannon Entropy across the whole MSA"""

    shannon_entropy_list = []
    for col_no in xrange(len(list(alignment[0]))):
        list_input = list(alignment[:, col_no])
        shannon_entropy_list.append(shannon_entropy(list_input))

    return shannon_entropy_list


def plot(index, sel, verbose):
    """"Create a quick plot via matplotlib to visualise"""
    import matplotlib.pyplot as plt

    if verbose > 0: print("Plotting data...")

    plt.plot(index, sel)
    plt.xlabel('MSA Position Index', fontsize=16)
    plt.ylabel('Shannon Entropy', fontsize=16)

    plt.show()


def running_mean(l, N):
    sum = 0
    result = list(0 for x in l)

    for i in range( 0, N ):
        sum = sum + l[i]
        result[i] = sum / (i+1)

    for i in range( N, len(l) ):
        sum = sum - l[i-N] + l[i]
        result[i] = sum / N

    return result

def main():
    """Compute Shannon Entropy from a provided MSA."""

    # Parse arguments
    args = parseArgs()

    # Convert object elements to standard variables for functions
    msa = args.alignment
    alnformat = args.alnformat
    verbose = args.verbose
    makeplot = args.makeplot
    runningmean = args.runningmean

# Start calling functions to do the heavy lifting

    alignment, seq_lengths, index = parseMSA(msa, alnformat, verbose)
    sel = shannon_entropy_list_msa(alignment)

    if runningmean > 0:
        sel = running_mean(sel, runningmean)

    if makeplot is True:
        plot(index, sel, verbose)

    if verbose > 0: print("Index" + '\t' + "Entropy")
    for c1, c2 in zip(index, sel):

        print(str(c1) + '\t' + str(c2))



if __name__ == '__main__':
    main()

In [ ]:
#!/usr/bin/env python3
import sys
import os
import subprocess
from collections import defaultdict
from Bio import SeqIO
from Bio import AlignIO
from math import log2
from ShannonMSA import shannon_entropy_list_msa, running_mean
import statistics
import argparse


def parse_args():
    parser = argparse.ArgumentParser(
        description="Expand hits around BLAST alignments and compute entropy"
    )

    # Positional (required) arguments
    parser.add_argument("query", help="Query FASTA file")
    parser.add_argument("genome", help="Genome FASTA file")
    parser.add_argument("bp_extend", type=int, help="Base pairs to extend")

    # Optional flags with default values
    parser.add_argument(
        "--outdir",
        default="results",
        help="Output directory (default: results)",
    )
    parser.add_argument(
        "-ec",
        type=float,
        default=1.2,
        help="Entropy cutoff for stopping expansion (default: 1.2)",
    )
    parser.add_argument(
        "--max_cycles",
        type=int,
        default=10,
        help="Maximum number of expansion cycles (default: 10)",
    )
    parser.add_argument(
        "--window",
        type=int,
        default=20,
        help="Window size for running mean entropy (default: 20)",
    )

    return parser.parse_args()


class Query:
    def __init__(self, record, db_file, exp, genome_file, out_dir="results"):
        self.record = record.id
        self.db_file = db_file
        self.exp = exp
        self.expand_cycles = 1
        self.query = f"{record.id}.fasta"
        self.results_tsv = os.path.join(out_dir, f"hits_{self.record}.tsv")
        self.results_bed = os.path.join(
            out_dir, f"hits_{self.record}_round{self.expand_cycles}.bed"
        )
        self.genome_file = genome_file
        self.fasta_exp_hits = os.path.join(
            out_dir, f"{self.record}_exp_hits_round{self.expand_cycles}.fa"
        )
        self.alignment = os.path.join(
            out_dir, f"{self.record}_round{self.expand_cycles}.afa"
        )
        self.entropy = None
        self.out_dir = out_dir

    def run_blast(self):
        print(f"Running BLAST for {self.record}...")
        os.makedirs(self.out_dir, exist_ok=True)

        subprocess.run(
            [
                "blastn",
                "-query",
                self.query,
                "-db",
                self.db_file,
                "-outfmt",
                "6 qseqid sseqid sstart send",
                "-out",
                self.results_tsv,
            ]
        )

    def expand_results(self):
        print(f"Round: {self.expand_cycles}. Expanding hits...")

        with open(self.results_tsv, "r") as fin, open(
            self.results_bed, "w"
        ) as fout:
            for line in fin:
                qseqid, sseqid, sstart, send = line.strip().split()
                sstart, send = int(sstart), int(send)

                # normalize strand orientation
                start, end = min(sstart, send), max(sstart, send)

                # expand window depending on cycle
                expand_size = self.exp * self.expand_cycles
                start = max(0, start - expand_size)  # avoid negative coords
                end = end + expand_size

                # BED requires 0-based start
                fout.write(f"{sseqid}\t{start - 1}\t{end}\n")

        self.expand_cycles += 1

        subprocess.run(
            [
                "bedtools",
                "getfasta",
                "-fi",
                self.genome_file,
                "-bed",
                self.results_bed,
                "-fo",
                self.fasta_exp_hits,
            ]
        )

    def mult_align(self, out_dir="results"):
        print("Aligning copies...")
        input_aln = self.fasta_exp_hits
        if os.path.isfile(input_aln):
            subprocess.run(
                ["muscle", "-align", input_aln, "-output", self.alignment, "--quiet"]
            )
        else:
            print("Error: no input file for MSA")

    def compute_alignment_score(self, msa_file, fmt="fasta", window=20):
        alignment = AlignIO.read(msa_file, fmt)
        entropies = shannon_entropy_list_msa(alignment)

        if window > 0:
            entropies = running_mean(entropies, window)

        self.entropy = statistics.median(entropies)
        print(
            f"Entropy for alignment round {self.expand_cycles} is {self.entropy}"
        )

        return self.entropy


def main():
    args = parse_args()
    query_file = args.query
    genome_file = args.genome
    number_of_bp_to_extend = args.bp_extend
    out_dir = args.outdir
    entropy_cutoff = args.ec
    max_cycles = args.max_cycles
    window = args.window

    work_d = os.getcwd()
    genome_prefix = os.path.splitext(genome_file)[0]

    if not os.path.isfile(os.path.join(work_d, genome_file)):
        print("Genome file is missing")
    elif not os.path.isfile(os.path.join(work_d, query_file)):
        print("Query file missing")
    else:
        subprocess.run(
            [
                "makeblastdb",
                "-in",
                genome_file,
                "-dbtype",
                "nucl",
                "-out",
                genome_prefix,
            ]
        )

        queries = []

        for record in SeqIO.parse(query_file, "fasta"):
            query_path = f"{record.id}.fasta"
            SeqIO.write(record, query_path, "fasta")

            q = Query(
                record,
                genome_prefix,
                number_of_bp_to_extend,
                genome_file,
                out_dir,
            )
            q.run_blast()
            queries.append(q)

            print(f"Results for {record.id} written to {q.results_tsv}")

            # Initial entropy calculation
            q.expand_results()
            q.mult_align()

            if os.path.exists(q.alignment):
                q.compute_alignment_score(q.alignment)

            while q.expand_cycles < 10 and (
                q.entropy is None or q.entropy > entropy_cutoff
            ):
                q.expand_results()
                print(f"Results for {record.id} written to {q.results_bed}")
                q.mult_align()
                print(f"Results for {record.id} written to {q.alignment}")

                if os.path.exists(q.alignment):
                    q.compute_alignment_score(q.alignment)
                    print(f"Entropy for {record.id}: {q.entropy}")
                    if q.entropy >= entropy_cutoff:
                        print(
                            "Expansion will stop due to entropy value reaching cutoff"
                        )


if __name__ == "__main__":
    main()


In [ ]:
name = ">JAEIGE010000253.1:1565971-1570570"

parts = name.strip().split(">")[1].split(":")
print(parts)


['JAEIGE010000253.1', ['1565971', '1570570']]
